In [1]:
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
import seaborn as sns

# ML
import mlflow
import mlflow.sklearn
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import classification_report, accuracy_score, f1_score, confusion_matrix
from sklearn.pipeline import Pipeline

# NLP
import nltk
nltk.download('vader_lexicon', quiet=True)
nltk.download('stopwords', quiet=True)
from nltk.sentiment.vader import SentimentIntensityAnalyzer
from nltk.corpus import stopwords


In [3]:
import pandas as pd
import numpy as np
import mlflow
import mlflow.sklearn
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score, f1_score
import re

# set up mlflow
mlflow.set_tracking_uri("sqlite:///mlruns.db")
mlflow.set_experiment("customer-support-intelligence")

print("MLflow setup done")
print(f"Tracking URI: {mlflow.get_tracking_uri()}")

MLflow setup done
Tracking URI: sqlite:///mlruns.db


In [4]:
merged_with_text = pd.read_csv("../data/raw/merged_with_text.csv")
df = pd.read_csv("../data/raw/twcs_engineered.csv")

print(f"df: {df.shape}")
print(f"merged_with_text: {merged_with_text.shape}")

df: (2811774, 13)
merged_with_text: (1261885, 10)


In [5]:
df.head(3)

,tweet_id,author_id,inbound,created_at,text,response_tweet_id,in_response_to_tweet_id,hour,day_of_week,date,month,text_length,month_str
0,1,sprintcare,False,2017-10-31 22:10:47+00:00,@115712 I understand. I would like to assist y...,2,3.0,22,Tuesday,2017-10-31,2017-10,121,2017-10
1,2,115712,True,2017-10-31 22:11:45+00:00,@sprintcare and how do you propose we do that,NaN,1.0,22,Tuesday,2017-10-31,2017-10,45,2017-10
2,3,115712,True,2017-10-31 22:08:27+00:00,@sprintcare I have sent several private messag...,1,4.0,22,Tuesday,2017-10-31,2017-10,82,2017-10


## Feature Engineering

### Decisions from EDA:
- Extract caps_ratio, emoji, punctuation features BEFORE cleaning (signals urgency)
- Clean text AFTER (remove @mentions, URLs, lowercase) for TF-IDF
- Cap response_time_mins at 500 — threading errors create impossible values (max 2.6M mins)
- Apply log transform to response_time after capping — right skewed distribution
- Will NOT use tweet_length — near-zero correlation with response time confirmed in EDA

In [6]:
# keep raw text as is — needed for sentiment, caps, emoji features
# create clean version separately for TF-IDF

def extract_raw_features(text):
    text = str(text)
    # caps ratio — before any cleaning
    letters = [c for c in text if c.isalpha()]
    caps_ratio = sum(1 for c in letters if c.isupper()) / max(len(letters), 1)
    
    # emoji detection
    emoji_pattern = re.compile("["
        u"\U0001F600-\U0001F64F"
        u"\U0001F300-\U0001F5FF"
        u"\U0001F680-\U0001F9FF"
        u"\U00002700-\U000027BF"
        "]+", flags=re.UNICODE)
    has_emoji = int(bool(emoji_pattern.search(text)))
    
    return {
        "caps_ratio": round(caps_ratio, 3),
        "has_emoji": has_emoji,
        "has_exclamation": int("!" in text),
        "has_question": int("?" in text),
        "exclamation_count": text.count("!"),
        "question_count": text.count("?"),
        "text_length": len(text)
    }

def clean_text(text):
    text = re.sub(r'@\w+', '', str(text))   # remove @mentions
    text = re.sub(r'http\S+', '', text)      # remove URLs
    text = re.sub(r'[^\w\s!?]', '', text)   # remove special chars but keep !?
    text = text.lower().strip()
    return text

In [7]:
# apply to customer tweets only
customer_df = df[df["inbound"]==True].copy()

# extract raw features first
print("Extracting raw features...")
raw_features = customer_df["text"].apply(extract_raw_features)
feature_df = pd.DataFrame(raw_features.tolist(), index=customer_df.index)
customer_df = pd.concat([customer_df, feature_df], axis=1)

# clean text 
customer_df["text_raw"] = customer_df["text"]       # original
customer_df["text_clean"] = customer_df["text"].apply(clean_text)  # for TF-IDF

print(f"Shape: {customer_df.shape}")
print(customer_df[["text_raw", "text_clean", "caps_ratio", "has_emoji", "has_exclamation"]].head(3))

Extracting raw features...
Shape: (1537843, 22)
                                            text_raw  \
1      @sprintcare and how do you propose we do that   
2  @sprintcare I have sent several private messag...   
4                                 @sprintcare I did.   

                                          text_clean  caps_ratio  has_emoji  \
1                  and how do you propose we do that       0.000          0   
2  i have sent several private messages and no on...       0.015          0   
4                                              i did       0.071          0   

   has_exclamation  
1                0  
2                0  
4                0  


In [10]:
#create the sentimnet 
from nltk.sentiment.vader import SentimentIntensityAnalyzer

sia = SentimentIntensityAnalyzer()

def get_sentiment(text):
    score = sia.polarity_scores(str(text))["compound"]
    if score <= -0.3:
        return "negative"
    else:
        return "non-negative"

customer_df["sentiment"] = customer_df["text_raw"].apply(get_sentiment)

print("Sentiment distribution:")
print(customer_df["sentiment"].value_counts())
print(f"\n{customer_df['sentiment'].value_counts(normalize=True).round(3)}")

Sentiment distribution:
sentiment
non-negative    1199149
negative         338694
Name: count, dtype: int64

sentiment
non-negative    0.78
negative        0.22
Name: proportion, dtype: float64


In [11]:
print("NEGATIVE examples:")
print(customer_df[customer_df["sentiment"]=="negative"]["text_raw"].sample(4, random_state=42).values)

print("\nnon-negative examples:")
print(customer_df[customer_df["sentiment"]=="non-negative"]["text_raw"].sample(4, random_state=42).values)

NEGATIVE examples:
["@British_Airways No, haven't seen a message with confirmation number. Would there be a delay in receiving confirmation due to earlier system outage?"
 'Finally what the fuck is up with these question marks @115858'
 "Well now that @tesco stretford metro closed down seems @AldiUK actually has lot of customers now. they may need to unblock the 2 blocked off tills as every time I've been, only 1 to 2 tills are open and 2 have products/baskets blocking them"
 '@115850 @AmazonHelp \nworst service..my product to be deliver before diwali but still not deliver, expected date already over.']

non-negative examples:
['@SpotifyCares Hi, I cant find an Indosat option to buy spotify premium.'
 '@AmericanAir How can a plane that has been sitting at the airport since 6pm yesterday not be check for engine issues until the plane is ready to taxi on the runway for takeoff at 11:20am!?!?  Thanks, now I am missing my connecting flight to get to my grandfather’s funeral...'
 "@SpotifyC

In [12]:
customer_df.to_csv("../data/raw/customer_features.csv", index=False)
print(f"Saved customer_features: {customer_df.shape}")

Saved customer_features: (1537843, 23)


## Sentiment model 

In [13]:
from sklearn.pipeline import Pipeline

X = customer_df["text_clean"]
y = customer_df["sentiment"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train: {len(X_train):,} | Test: {len(X_test):,}")
print(f"\nTrain distribution:")
print(y_train.value_counts(normalize=True).round(3))

Train: 1,230,274 | Test: 307,569

Train distribution:
sentiment
non-negative    0.78
negative        0.22
Name: proportion, dtype: float64


In [14]:
with mlflow.start_run(run_name="logistic_regression_tfidf"):
    
    # params
    max_features = 5000
    C = 1.0
    
    # log params
    mlflow.log_param("model", "LogisticRegression")
    mlflow.log_param("max_features", max_features)
    mlflow.log_param("C", C)
    mlflow.log_param("class_weight", "balanced")
    
    # pipeline, TF-IDF fit on train only
    pipeline = Pipeline([
        ("tfidf", TfidfVectorizer(max_features=max_features)),
        ("clf", LogisticRegression(C=C, max_iter=1000, 
                                   class_weight="balanced", 
                                   random_state=42))
    ])
    
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)
    
    # metrics
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average="weighted")
    f1_neg = f1_score(y_test, y_pred, pos_label="negative", average="binary")
    
    # log metrics
    mlflow.log_metric("accuracy", acc)
    mlflow.log_metric("f1_weighted", f1)
    mlflow.log_metric("f1_negative", f1_neg)
    
    # log model
    mlflow.sklearn.log_model(pipeline, "model")
    
    print(classification_report(y_test, y_pred))
    print(f"Accuracy: {acc:.3f}")
    print(f"F1 weighted: {f1:.3f}")
    print(f"F1 negative class: {f1_neg:.3f}")

2026/06/23 16:28:46 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/23 16:28:56 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


              precision    recall  f1-score   support

    negative       0.68      0.87      0.76     67739
non-negative       0.96      0.88      0.92    239830

    accuracy                           0.88    307569
   macro avg       0.82      0.88      0.84    307569
weighted avg       0.90      0.88      0.88    307569

Accuracy: 0.879
F1 weighted: 0.884
F1 negative class: 0.760


In [15]:
with mlflow.start_run(run_name="random_forest_tfidf"):
    
    mlflow.log_param("model", "RandomForest")
    mlflow.log_param("max_features", 5000)
    mlflow.log_param("n_estimators", 100)
    mlflow.log_param("class_weight", "balanced")
    
    pipeline_rf = Pipeline([
        ("tfidf", TfidfVectorizer(max_features=5000)),
        ("clf", RandomForestClassifier(n_estimators=100, 
                                       class_weight="balanced",
                                       random_state=42,
                                       n_jobs=-1))
    ])
    
    pipeline_rf.fit(X_train, y_train)
    y_pred_rf = pipeline_rf.predict(X_test)
    
    acc_rf = accuracy_score(y_test, y_pred_rf)
    f1_rf = f1_score(y_test, y_pred_rf, average="weighted")
    f1_neg_rf = f1_score(y_test, y_pred_rf, pos_label="negative", average="binary")
    
    mlflow.log_metric("accuracy", acc_rf)
    mlflow.log_metric("f1_weighted", f1_rf)
    mlflow.log_metric("f1_negative", f1_neg_rf)
    mlflow.sklearn.log_model(pipeline_rf, "model")
    
    print(classification_report(y_test, y_pred_rf))
    print(f"Accuracy: {acc_rf:.3f}")
    print(f"F1 weighted: {f1_rf:.3f}")
    print(f"F1 negative class: {f1_neg_rf:.3f}")